In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/syedjaffarrazakazmi/smart-om/SMART-OM/Metadata/Patient_s Metadata.xlsx
/kaggle/input/datasets/syedjaffarrazakazmi/smart-om/SMART-OM/03. OPMD/04. Lesion annotation/02. Ventral tongue/SMITA00187_W_VT.jpeg
/kaggle/input/datasets/syedjaffarrazakazmi/smart-om/SMART-OM/03. OPMD/04. Lesion annotation/05. Upper lip/SMITA00187_W_UL.jpeg
/kaggle/input/datasets/syedjaffarrazakazmi/smart-om/SMART-OM/03. OPMD/04. Lesion annotation/03. Left buccal mucosa/SMITA00274_W_LB.jpg
/kaggle/input/datasets/syedjaffarrazakazmi/smart-om/SMART-OM/03. OPMD/04. Lesion annotation/03. Left buccal mucosa/SMITA000233_R_LB2.jpeg
/kaggle/input/datasets/syedjaffarrazakazmi/smart-om/SMART-OM/03. OPMD/04. Lesion annotation/03. Left buccal mucosa/SMITA00262_R_LB.jpeg
/kaggle/input/datasets/syedjaffarrazakazmi/smart-om/SMART-OM/03. OPMD/04. Lesion annotation/03. Left buccal mucosa/SMITA00034_R_LB.JPG
/kaggle/input/datasets/syedjaffarrazakazmi/smart-om/SMART-OM/03. OPMD/04. Lesion annotation/03. Left bu

In [2]:
import json
import pandas as pd
from pathlib import Path

data_dir = Path('/kaggle/input/datasets/syedjaffarrazakazmi/smart-om')  
for item in sorted(data_dir.rglob('*')):
    if item.is_file():
        print(item)

/kaggle/input/datasets/syedjaffarrazakazmi/smart-om/SMART-OM/01. Normal/01. Unannotated/01. Dorsal tongue/5 - DT.jpg
/kaggle/input/datasets/syedjaffarrazakazmi/smart-om/SMART-OM/01. Normal/01. Unannotated/01. Dorsal tongue/6 - DT.jpg
/kaggle/input/datasets/syedjaffarrazakazmi/smart-om/SMART-OM/01. Normal/01. Unannotated/01. Dorsal tongue/7 - DT.jpg
/kaggle/input/datasets/syedjaffarrazakazmi/smart-om/SMART-OM/01. Normal/01. Unannotated/01. Dorsal tongue/8 - DT.jpg
/kaggle/input/datasets/syedjaffarrazakazmi/smart-om/SMART-OM/01. Normal/01. Unannotated/01. Dorsal tongue/9 - DT.jpg
/kaggle/input/datasets/syedjaffarrazakazmi/smart-om/SMART-OM/01. Normal/01. Unannotated/01. Dorsal tongue/SMITA00001_W_DT.jpeg
/kaggle/input/datasets/syedjaffarrazakazmi/smart-om/SMART-OM/01. Normal/01. Unannotated/01. Dorsal tongue/SMITA00004_W_DT.jpeg
/kaggle/input/datasets/syedjaffarrazakazmi/smart-om/SMART-OM/01. Normal/01. Unannotated/01. Dorsal tongue/SMITA00005_W_DT.jpg
/kaggle/input/datasets/syedjaffarra

In [3]:
from pathlib import Path
from PIL import Image
import pandas as pd
import re

DATA_DIR = Path('/kaggle/input/datasets/syedjaffarrazakazmi/smart-om')
DATASET_NAME = 'smart_om'


valid_extensions = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG'}

image_records = []

# Scan all image files
for file_path in DATA_DIR.rglob('*'):
    if file_path.is_file() and file_path.suffix in valid_extensions:
        parts = file_path.parts
        
        
        condition = parts[6] if len(parts) > 6 else 'Unknown'       # e.g., '01. Normal'
        annotation_group = parts[7] if len(parts) > 7 else 'Unknown' # e.g., '01. Unannotated'
        anatomical_site = parts[8] if len(parts) > 8 else 'Unknown'  # e.g., '01. Dorsal tongue'
        
        filename = file_path.name
        stem = file_path.stem
        
        # Extract Patient ID (e.g., SMITA00001 from SMITA00001_W_DT)
        patient_match = re.match(r'^(SMITA\d+|\d+)', stem)
        patient_id = patient_match.group(1) if patient_match else 'Unknown'

        # Extract capture/light type if present (e.g., _W_ for White light, _R_ for Regular)
        capture_type = 'Unknown'
        if '_W_' in filename or '_W-' in filename:
            capture_type = 'White Light'
        elif '_R_' in filename or '_R-' in filename:
            capture_type = 'Regular'

        image_records.append({
            'source_dataset': DATASET_NAME,
            'file_name': filename,
            'file_path': str(file_path),
            'image_stem': stem,
            'patient_id': patient_id,
            'condition_category': condition,
            'annotation_group': annotation_group,
            'anatomical_site': anatomical_site,
            'capture_type': capture_type
        })

df_images = pd.DataFrame(image_records)
print(f"Total images cataloged: {len(df_images)}")
print(df_images.head())

Total images cataloged: 7731
  source_dataset               file_name  \
0       smart_om    SMITA00187_W_VT.jpeg   
1       smart_om    SMITA00187_W_UL.jpeg   
2       smart_om     SMITA00274_W_LB.jpg   
3       smart_om  SMITA000233_R_LB2.jpeg   
4       smart_om    SMITA00262_R_LB.jpeg   

                                           file_path         image_stem  \
0  /kaggle/input/datasets/syedjaffarrazakazmi/sma...    SMITA00187_W_VT   
1  /kaggle/input/datasets/syedjaffarrazakazmi/sma...    SMITA00187_W_UL   
2  /kaggle/input/datasets/syedjaffarrazakazmi/sma...    SMITA00274_W_LB   
3  /kaggle/input/datasets/syedjaffarrazakazmi/sma...  SMITA000233_R_LB2   
4  /kaggle/input/datasets/syedjaffarrazakazmi/sma...    SMITA00262_R_LB   

    patient_id condition_category annotation_group        anatomical_site  \
0   SMITA00187           SMART-OM         03. OPMD  04. Lesion annotation   
1   SMITA00187           SMART-OM         03. OPMD  04. Lesion annotation   
2   SMITA00274          

In [4]:
widths = []
heights = []
is_valid = []

print("Extracting image resolutions and verifying files...")

for idx, row in df_images.iterrows():
    try:
        with Image.open(row['file_path']) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
            is_valid.append(True)
    except Exception as e:
        widths.append(None)
        heights.append(None)
        is_valid.append(False)

df_images['width'] = widths
df_images['height'] = heights
df_images['is_valid_image'] = is_valid

print(f"Done! Valid images: {df_images['is_valid_image'].sum()} / {len(df_images)}")
print("\n--- Image Dimensions Summary ---")
print(df_images[['width', 'height']].describe())

Extracting image resolutions and verifying files...
Done! Valid images: 7731 / 7731

--- Image Dimensions Summary ---
             width       height
count  7731.000000  7731.000000
mean    908.133101   708.906480
std     382.565627   314.918927
min     211.000000   143.000000
25%     594.000000   519.000000
50%     870.000000   599.000000
75%    1060.500000   815.500000
max    2268.000000  4032.000000


In [5]:
import json
import pandas as pd
from pathlib import Path

annotation_records = []
json_files = list(DATA_DIR.rglob('*.json'))

print(f"Scanning and parsing {len(json_files)} VIA JSON files...")

for json_path in json_files:
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        img_metadata = data.get('_via_img_metadata', {})
        
        for via_id, entry in img_metadata.items():
            filename = entry.get('filename', '')
            stem = Path(filename).stem if filename else json_path.stem.replace('_Lesion', '').replace('_region', '').replace('_full', '')
            
            regions = entry.get('regions', [])
            for region in regions:
                shape = region.get('shape_attributes', {})
                region_attr = region.get('region_attributes', {})
                
                shape_type = shape.get('name', 'unknown')
                points_x = shape.get('all_points_x', [])
                points_y = shape.get('all_points_y', [])
                
                # Compute bounding box [xmin, ymin, xmax, ymax] and area
                xmin = min(points_x) if points_x else None
                xmax = max(points_x) if points_x else None
                ymin = min(points_y) if points_y else None
                ymax = max(points_y) if points_y else None
                bbox_width = (xmax - xmin) if (xmin is not None and xmax is not None) else None
                bbox_height = (ymax - ymin) if (ymin is not None and ymax is not None) else None
                
                # Determine region label
                label = 'lesion_or_region'
                if region_attr:
                    label = next(iter(region_attr.values()), label)
                
                annotation_records.append({
                    'json_file': json_path.name,
                    'json_path': str(json_path),
                    'image_filename': filename,
                    'image_stem': stem,
                    'shape_type': shape_type,
                    'num_points': len(points_x),
                    'label': label,
                    'xmin': xmin,
                    'ymin': ymin,
                    'xmax': xmax,
                    'ymax': ymax,
                    'bbox_width': bbox_width,
                    'bbox_height': bbox_height
                })
    except Exception as e:
        print(f"Error reading {json_path.name}: {e}")

df_annotations = pd.DataFrame(annotation_records)

print(f"\nDone! Extracted {len(df_annotations)} region annotations.")
print(df_annotations.head())

Scanning and parsing 1058 VIA JSON files...
Error reading SMITA00212_R_full.json: 'NoneType' object has no attribute 'get'
Error reading SMITA00028_R_region.json: 'NoneType' object has no attribute 'get'
Error reading SMITA00028_R_full.json: 'NoneType' object has no attribute 'get'

Done! Extracted 37943 region annotations.
                   json_file  \
0  SMITA000252_R_Lesion.json   
1   SMITA00010_W_Lesion.json   
2   SMITA00010_W_Lesion.json   
3   SMITA00262_R_Lesion.json   
4   SMITA00262_R_Lesion.json   

                                           json_path         image_filename  \
0  /kaggle/input/datasets/syedjaffarrazakazmi/sma...  SMITA000252_R_LB.jpeg   
1  /kaggle/input/datasets/syedjaffarrazakazmi/sma...    SMITA00010_W_LB.jpg   
2  /kaggle/input/datasets/syedjaffarrazakazmi/sma...    SMITA00010_W_LB.jpg   
3  /kaggle/input/datasets/syedjaffarrazakazmi/sma...   SMITA00262_R_LB.jpeg   
4  /kaggle/input/datasets/syedjaffarrazakazmi/sma...  SMITA00262_R_RB1.jpeg   

      

In [6]:
# Aggregate annotation metrics per image stem
if not df_annotations.empty:
    ann_summary = df_annotations.groupby('image_stem').agg(
        annotation_count=('shape_type', 'count'),
        total_annotated_points=('num_points', 'sum')
    ).reset_index()
    
    # Merge back into the image metadata DataFrame
    standardized_df = df_images.merge(ann_summary, on='image_stem', how='left')
    standardized_df['annotation_count'] = standardized_df['annotation_count'].fillna(0).astype(int)
    standardized_df['total_annotated_points'] = standardized_df['total_annotated_points'].fillna(0).astype(int)
else:
    standardized_df = df_images.copy()
    standardized_df['annotation_count'] = 0
    standardized_df['total_annotated_points'] = 0

print(f"Final dataset rows: {len(standardized_df)}")
print(f"Annotated images: {(standardized_df['annotation_count'] > 0).sum()} / {len(standardized_df)}")

print("\n--- Distribution by Condition Category ---")
print(standardized_df['condition_category'].value_counts())

print("\n--- Distribution by Anatomical Site ---")
print(standardized_df['anatomical_site'].value_counts())

print("\n--- Standardized Dataset Preview ---")
print(standardized_df.head())

Final dataset rows: 7731
Annotated images: 7015 / 7731

--- Distribution by Condition Category ---
condition_category
SMART-OM    7731
Name: count, dtype: int64

--- Distribution by Anatomical Site ---
anatomical_site
02. Region annotation    2469
03. Full annotation      2469
01. Unannotated          2469
04. Lesion annotation     324
Name: count, dtype: int64

--- Standardized Dataset Preview ---
  source_dataset               file_name  \
0       smart_om    SMITA00187_W_VT.jpeg   
1       smart_om    SMITA00187_W_UL.jpeg   
2       smart_om     SMITA00274_W_LB.jpg   
3       smart_om  SMITA000233_R_LB2.jpeg   
4       smart_om    SMITA00262_R_LB.jpeg   

                                           file_path         image_stem  \
0  /kaggle/input/datasets/syedjaffarrazakazmi/sma...    SMITA00187_W_VT   
1  /kaggle/input/datasets/syedjaffarrazakazmi/sma...    SMITA00187_W_UL   
2  /kaggle/input/datasets/syedjaffarrazakazmi/sma...    SMITA00274_W_LB   
3  /kaggle/input/datasets/syedjaf

In [7]:
# 1. Reassign condition and annotation group accurately
standardized_df['clinical_condition'] = standardized_df['annotation_group']
standardized_df['annotation_stage'] = standardized_df['anatomical_site']

# 2. Map anatomical site codes from file stems
site_code_map = {
    'DT': 'Dorsal tongue',
    'VT': 'Ventral tongue',
    'LB': 'Left buccal mucosa',
    'RB': 'Right buccal mucosa',
    'UL': 'Upper lip',
    'LL': 'Lower lip',
    'UA': 'Upper arch',
    'LA': 'Lower arch'
}

def extract_site(stem):
    for code, full_name in site_code_map.items():
        if f'_{code}' in stem or f'-{code}' in stem or stem.endswith(code):
            return full_name
    return 'Other'

standardized_df['oral_site'] = standardized_df['image_stem'].apply(extract_site)

# Drop redundant raw columns
columns_to_keep = [
    'source_dataset', 'file_name', 'file_path', 'image_stem', 'patient_id',
    'clinical_condition', 'annotation_stage', 'oral_site', 'capture_type',
    'width', 'height', 'is_valid_image', 'annotation_count', 'total_annotated_points'
]
standardized_df = standardized_df[columns_to_keep]

print("--- Clinical Conditions ---")
print(standardized_df['clinical_condition'].value_counts())

print("\n--- Oral Sites ---")
print(standardized_df['oral_site'].value_counts())

print("\n--- Annotation Stages ---")
print(standardized_df['annotation_stage'].value_counts())

--- Clinical Conditions ---
clinical_condition
01. Normal                   6435
02. Variation from normal     716
03. OPMD                      500
04. Oral Cancer                80
Name: count, dtype: int64

--- Oral Sites ---
oral_site
Left buccal mucosa     1094
Right buccal mucosa    1088
Dorsal tongue          1010
Lower lip               976
Ventral tongue          906
Upper arch              876
Lower arch              867
Upper lip               826
Other                    88
Name: count, dtype: int64

--- Annotation Stages ---
annotation_stage
02. Region annotation    2469
03. Full annotation      2469
01. Unannotated          2469
04. Lesion annotation     324
Name: count, dtype: int64


In [8]:
from pathlib import Path

# Paths for exported CSVs
meta_csv_path = Path('/kaggle/working/smart_om_standardized_images.csv')
ann_csv_path = Path('/kaggle/working/smart_om_polygon_annotations.csv')

# Export DataFrames
standardized_df.to_csv(meta_csv_path, index=False)
df_annotations.to_csv(ann_csv_path, index=False)

# Validation Checks
assert len(standardized_df) == 7731, "Total row count mismatch"
assert standardized_df['is_valid_image'].all(), "Corrupted images detected"
assert standardized_df['patient_id'].isna().sum() == 0, "Missing patient IDs"

print(f"Standardized metadata exported to: {meta_csv_path}")
print(f"Polygon annotations exported to: {ann_csv_path}")
print(f"Total images: {len(standardized_df)} | Unique patients: {standardized_df['patient_id'].nunique()}")
print("\nFinal Metadata Preview:")
print(standardized_df.head())

Standardized metadata exported to: /kaggle/working/smart_om_standardized_images.csv
Polygon annotations exported to: /kaggle/working/smart_om_polygon_annotations.csv
Total images: 7731 | Unique patients: 320

Final Metadata Preview:
  source_dataset               file_name  \
0       smart_om    SMITA00187_W_VT.jpeg   
1       smart_om    SMITA00187_W_UL.jpeg   
2       smart_om     SMITA00274_W_LB.jpg   
3       smart_om  SMITA000233_R_LB2.jpeg   
4       smart_om    SMITA00262_R_LB.jpeg   

                                           file_path         image_stem  \
0  /kaggle/input/datasets/syedjaffarrazakazmi/sma...    SMITA00187_W_VT   
1  /kaggle/input/datasets/syedjaffarrazakazmi/sma...    SMITA00187_W_UL   
2  /kaggle/input/datasets/syedjaffarrazakazmi/sma...    SMITA00274_W_LB   
3  /kaggle/input/datasets/syedjaffarrazakazmi/sma...  SMITA000233_R_LB2   
4  /kaggle/input/datasets/syedjaffarrazakazmi/sma...    SMITA00262_R_LB   

    patient_id clinical_condition       annotation_

In [9]:
# Step 6 — Class distribution & Annotation categories
print("=== Annotation Shape Types ===")
print(df_annotations['shape_type'].value_counts())

print("\n=== Annotation Labels ===")
print(df_annotations['label'].value_counts())

print("\n=== Clinical Conditions ===")
print(standardized_df['clinical_condition'].value_counts())

print("\n=== Oral Anatomical Sites ===")
print(standardized_df['oral_site'].value_counts())

print("\n=== Capture Type (Lighting) ===")
print(standardized_df['capture_type'].value_counts())

=== Annotation Shape Types ===
shape_type
polygon     21979
rect        15116
polyline      488
ellipse       274
circle         86
Name: count, dtype: int64

=== Annotation Labels ===
label
lesion_or_region    37943
Name: count, dtype: int64

=== Clinical Conditions ===
clinical_condition
01. Normal                   6435
02. Variation from normal     716
03. OPMD                      500
04. Oral Cancer                80
Name: count, dtype: int64

=== Oral Anatomical Sites ===
oral_site
Left buccal mucosa     1094
Right buccal mucosa    1088
Dorsal tongue          1010
Lower lip               976
Ventral tongue          906
Upper arch              876
Lower arch              867
Upper lip               826
Other                    88
Name: count, dtype: int64

=== Capture Type (Lighting) ===
capture_type
Regular        6139
White Light    1405
Unknown         187
Name: count, dtype: int64


In [10]:
# Step 7 — Patient-level distribution
patient_img_counts = standardized_df.groupby('patient_id').size()
print("Images per patient — min/max/median:",
      patient_img_counts.min(), patient_img_counts.max(), patient_img_counts.median())

print(f"\nTotal Unique Patients: {standardized_df['patient_id'].nunique()}")

print("\nPatients per Clinical Condition:")
print(standardized_df.groupby('clinical_condition')['patient_id'].nunique())

print("\nPatients per Oral Site:")
print(standardized_df.groupby('oral_site')['patient_id'].nunique())

Images per patient — min/max/median: 3 57 24.0

Total Unique Patients: 320

Patients per Clinical Condition:
clinical_condition
01. Normal                   306
02. Variation from normal    112
03. OPMD                      65
04. Oral Cancer                9
Name: patient_id, dtype: int64

Patients per Oral Site:
oral_site
Dorsal tongue          303
Left buccal mucosa     297
Lower arch             281
Lower lip              288
Other                   12
Right buccal mucosa    297
Upper arch             277
Upper lip              268
Ventral tongue         282
Name: patient_id, dtype: int64


In [11]:
# Step 8 — Resolution (Width / Height)
widths  = standardized_df['width'].dropna().tolist()
heights = standardized_df['height'].dropna().tolist()

print("Width  — min/max/median:", min(widths), max(widths), pd.Series(widths).median())
print("Height — min/max/median:", min(heights), max(heights), pd.Series(heights).median())

print("\n--- Image Resolution Summary Statistics ---")
print(standardized_df[['width', 'height']].describe())

Width  — min/max/median: 211 2268 870.0
Height — min/max/median: 143 4032 599.0

--- Image Resolution Summary Statistics ---
             width       height
count  7731.000000  7731.000000
mean    908.133101   708.906480
std     382.565627   314.918927
min     211.000000   143.000000
25%     594.000000   519.000000
50%     870.000000   599.000000
75%    1060.500000   815.500000
max    2268.000000  4032.000000


In [12]:
# Step 9 — Out-of-bounds bounding box & polygon validation
# Map image stems to their corresponding width and height
dims_map = dict(zip(standardized_df['image_stem'], zip(standardized_df['width'], standardized_df['height'])))

bad_annotations = []
overflow_records = []

for _, ann in df_annotations.iterrows():
    stem = ann['image_stem']
    if stem in dims_map:
        w, h = dims_map[stem]
        if w is not None and h is not None and ann['xmin'] is not None:
            overflow_x = max(0, ann['xmax'] - w)
            overflow_y = max(0, ann['ymax'] - h)
            neg_x = min(0, ann['xmin'])
            neg_y = min(0, ann['ymin'])
            
            if ann['xmin'] < 0 or ann['ymin'] < 0 or ann['xmax'] > w or ann['ymax'] > h:
                bad_annotations.append(ann)
                overflow_records.append({
                    'image_stem': stem,
                    'json_file': ann['json_file'],
                    'overflow_x': overflow_x,
                    'overflow_y': overflow_y,
                    'neg_x': neg_x,
                    'neg_y': neg_y
                })

print("Out-of-bounds annotations count:", len(bad_annotations))

if overflow_records:
    df_overflow = pd.DataFrame(overflow_records)
    print("Unique images affected:", df_overflow['image_stem'].nunique())
    print("\nOverflow stats (pixels):")
    print(df_overflow[['overflow_x', 'overflow_y', 'neg_x', 'neg_y']].describe())
else:
    print("All annotations fit within image dimensions.")

Out-of-bounds annotations count: 420
Unique images affected: 140

Overflow stats (pixels):
       overflow_x   overflow_y  neg_x       neg_y
count  420.000000   420.000000  420.0  420.000000
mean   100.097619    72.350000    0.0   -0.019048
std    184.549112   180.504646    0.0    0.181794
min      0.000000     0.000000    0.0   -2.000000
25%      0.000000     1.000000    0.0    0.000000
50%      0.000000     1.000000    0.0    0.000000
75%    170.000000     8.500000    0.0    0.000000
max    790.000000  1043.000000    0.0    0.000000


In [13]:
 # Step 10 — Check salvageability by swapping w and h
salvageable_stems = set()
truly_broken_stems = set()

affected_stems = set(df_overflow['image_stem'].unique())

for _, ann in df_annotations.iterrows():
    stem = ann['image_stem']
    if stem in affected_stems:
        w, h = dims_map[stem]
        if w is not None and h is not None and ann['xmin'] is not None:
            # Check if swapping w and h makes box valid (or within a 2px margin)
            if ann['xmin'] >= -2 and ann['ymin'] >= -2 and ann['xmax'] <= (h + 2) and ann['ymax'] <= (w + 2):
                salvageable_stems.add(stem)
            else:
                truly_broken_stems.add(stem)

print("Salvageable with rotation swap:", len(salvageable_stems - truly_broken_stems))
print("Requires coordinate clipping / edge adjustment:", len(truly_broken_stems))

Salvageable with rotation swap: 1
Requires coordinate clipping / edge adjustment: 139


In [14]:
# Step 11 — Process images into clean working directory
import shutil
from PIL import Image
from pathlib import Path

output_image_dir = Path('/kaggle/working/smart_om_clean_images')
output_image_dir.mkdir(parents=True, exist_ok=True)

rotated_count = 0
copied_count = 0

# Stems that need 90-deg rotation
rotate_stems = salvageable_stems - truly_broken_stems

for idx, row in standardized_df.iterrows():
    src_path = Path(row['file_path'])
    dst_path = output_image_dir / row['file_name']
    
    if row['image_stem'] in rotate_stems:
        with Image.open(src_path) as im:
            rotated = im.rotate(90, expand=True)
            rotated.save(dst_path)
            dims_map[row['image_stem']] = rotated.size
            standardized_df.at[idx, 'width'] = rotated.size[0]
            standardized_df.at[idx, 'height'] = rotated.size[1]
        rotated_count += 1
    else:
        shutil.copy2(src_path, dst_path)
        copied_count += 1

print(f"Rotated images: {rotated_count}")
print(f"Copied images: {copied_count}")
print(f"Total processed: {rotated_count + copied_count}")


# Clip bounding box coordinates with safe dictionary access
clipped_count = 0

for idx, ann in df_annotations.iterrows():
    stem = ann['image_stem']
    dims = dims_map.get(stem)
    
    if dims is not None and ann['xmin'] is not None:
        w, h = dims
        if w is not None and h is not None:
            df_annotations.at[idx, 'xmin'] = max(0, ann['xmin'])
            df_annotations.at[idx, 'ymin'] = max(0, ann['ymin'])
            df_annotations.at[idx, 'xmax'] = min(w, ann['xmax'])
            df_annotations.at[idx, 'ymax'] = min(h, ann['ymax'])
            df_annotations.at[idx, 'bbox_width'] = df_annotations.at[idx, 'xmax'] - df_annotations.at[idx, 'xmin']
            df_annotations.at[idx, 'bbox_height'] = df_annotations.at[idx, 'ymax'] - df_annotations.at[idx, 'ymin']
            clipped_count += 1

print(f"Successfully clipped and aligned {clipped_count} annotation boundaries.")

Rotated images: 4
Copied images: 7727
Total processed: 7731
Successfully clipped and aligned 37846 annotation boundaries.


In [15]:
# Step 12 — Out-of-bounds re-verification
bad_boxes_after = 0

for _, ann in df_annotations.iterrows():
    stem = ann['image_stem']
    dims = dims_map.get(stem)
    if dims is not None and ann['xmin'] is not None:
        w, h = dims
        if w is not None and h is not None:
            if ann['xmin'] < 0 or ann['ymin'] < 0 or ann['xmax'] > w or ann['ymax'] > h:
                bad_boxes_after += 1

print("Out-of-bounds annotations after fix:", bad_boxes_after)
assert bad_boxes_after == 0, "Fix incomplete: out-of-bounds coordinates remain."
print("Assertion Passed: All bounding coordinates are valid.")

Out-of-bounds annotations after fix: 0
Assertion Passed: All bounding coordinates are valid.


In [16]:
# Step 13 — Export and final sanity checks
from pathlib import Path

# Export cleaned metadata and annotations
export_meta_path = Path('/kaggle/working/smart_om_standardized.csv')
export_ann_path = Path('/kaggle/working/smart_om_polygon_annotations_cleaned.csv')

standardized_df.to_csv(export_meta_path, index=False)
df_annotations.to_csv(export_ann_path, index=False)

print("Exported metadata shape:", standardized_df.shape)
print("Exported annotations shape:", df_annotations.shape)


assert standardized_df['file_path'].nunique() == len(standardized_df), "Duplicate image file paths found"
assert standardized_df['clinical_condition'].isna().sum() == 0, "Unmatched clinical conditions"
assert standardized_df['patient_id'].isna().sum() == 0, "Unmatched patient IDs"
assert standardized_df['is_valid_image'].all(), "Corrupted images detected"

print("\nAll assertions passed successfully!")
print("Columns:", standardized_df.columns.tolist())
standardized_df.head(5)

Exported metadata shape: (7731, 14)
Exported annotations shape: (37943, 13)

All assertions passed successfully!
Columns: ['source_dataset', 'file_name', 'file_path', 'image_stem', 'patient_id', 'clinical_condition', 'annotation_stage', 'oral_site', 'capture_type', 'width', 'height', 'is_valid_image', 'annotation_count', 'total_annotated_points']


,source_dataset,file_name,file_path,image_stem,patient_id,clinical_condition,annotation_stage,oral_site,capture_type,width,height,is_valid_image,annotation_count,total_annotated_points
0,smart_om,SMITA00187_W_VT.jpeg,/kaggle/input/datasets/syedjaffarrazakazmi/sma...,SMITA00187_W_VT,SMITA00187,03. OPMD,04. Lesion annotation,Ventral tongue,White Light,385,410,True,14,103
1,smart_om,SMITA00187_W_UL.jpeg,/kaggle/input/datasets/syedjaffarrazakazmi/sma...,SMITA00187_W_UL,SMITA00187,03. OPMD,04. Lesion annotation,Upper lip,White Light,562,221,True,7,48
2,smart_om,SMITA00274_W_LB.jpg,/kaggle/input/datasets/syedjaffarrazakazmi/sma...,SMITA00274_W_LB,SMITA00274,03. OPMD,04. Lesion annotation,Left buccal mucosa,White Light,526,519,True,17,190
3,smart_om,SMITA000233_R_LB2.jpeg,/kaggle/input/datasets/syedjaffarrazakazmi/sma...,SMITA000233_R_LB2,SMITA000233,03. OPMD,04. Lesion annotation,Left buccal mucosa,Regular,635,599,True,25,925
4,smart_om,SMITA00262_R_LB.jpeg,/kaggle/input/datasets/syedjaffarrazakazmi/sma...,SMITA00262_R_LB,SMITA00262,03. OPMD,04. Lesion annotation,Left buccal mucosa,Regular,729,599,True,17,721
